# EagleVision ScanNet 3D-from-X Evaluation

This notebook runs inference-only evaluation for:

1. Frozen Depth Anything V2-Small depth
2. Depth Anything V2-Small + EagleVision residual adapter checkpoint

Attach your ScanNet-style RGB-D dataset and EagleVision checkpoint as Kaggle inputs. No training, DARES, LoRA, or SCARED path is used here.


## Expected Kaggle Inputs

Attach inputs under `/kaggle/input`:

- ScanNet-style data with scene folders containing `color/`, `depth/`, `pose/`, and `intrinsic/`.
- EagleVision adapter checkpoint, usually `best.pt`.
- Depth Anything V2-Small base checkpoint if your repo/config does not already point to one.

The notebook auto-detects likely paths, but you can override them in the next cell.


In [ ]:
from pathlib import Path
import os

KAGGLE_INPUT = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working')

# Optional manual overrides. Leave as None for auto-detection.
SCANNET_ROOT_OVERRIDE = None  # e.g. Path('/kaggle/input/scannet-2d/scannet')
ADAPTER_CHECKPOINT_OVERRIDE = None  # e.g. Path('/kaggle/input/my-weights/best.pt')
BASE_DA2_CHECKPOINT_OVERRIDE = None  # e.g. Path('/kaggle/input/da2-small/depth_anything_v2_vits.pth')
REPO_DIR_OVERRIDE = None  # e.g. Path('/kaggle/working/EagleVision')

# Evaluation budget knobs.
IMAGE_SIZE = [384, 512]
MAX_SCENES = 20
FRAMES_PER_SCENE = 20
MAX_FRAMES_PER_SCENE_FOR_MANIFEST = 200
FRAME_STRIDE = 5
MAX_PAIRS_PER_SCENE = 100
BATCH_SIZE = 4
NUM_WORKERS = 2
NUM_PREVIEWS = 16

OUT_ROOT = WORK_DIR / 'eaglevision_3d_from_x_scannet'
MANIFEST_DIR = OUT_ROOT / 'manifests' / 'scannet'
PRED_DIR = OUT_ROOT / 'predictions'
RAW_DIR = OUT_ROOT / 'raw'
AGG_DIR = OUT_ROOT / 'aggregated'
CONFIG_DIR = OUT_ROOT / 'configs'

for p in [OUT_ROOT, MANIFEST_DIR, PRED_DIR, RAW_DIR, AGG_DIR, CONFIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Kaggle input:', KAGGLE_INPUT)
print('Output root:', OUT_ROOT)


In [ ]:
import shutil, sys, subprocess

def looks_like_scannet_scene(path: Path) -> bool:
    return path.is_dir() and (path / 'color').exists() and (path / 'depth').exists() and (path / 'pose').exists() and (path / 'intrinsic').exists()

def find_scannet_root() -> Path:
    if SCANNET_ROOT_OVERRIDE is not None:
        return Path(SCANNET_ROOT_OVERRIDE)
    candidates = [KAGGLE_INPUT]
    candidates += [p for p in KAGGLE_INPUT.glob('*') if p.is_dir()]
    candidates += [p for p in KAGGLE_INPUT.glob('*/*') if p.is_dir()]
    for candidate in candidates:
        try:
            children = list(candidate.iterdir())[:200]
        except Exception:
            continue
        if any(looks_like_scannet_scene(child) for child in children):
            return candidate
    raise FileNotFoundError('Could not auto-detect ScanNet root. Set SCANNET_ROOT_OVERRIDE.')

def find_repo_dir() -> Path:
    if REPO_DIR_OVERRIDE is not None:
        return Path(REPO_DIR_OVERRIDE)
    candidates = [Path.cwd(), WORK_DIR / 'EagleVision']
    candidates += [p for p in KAGGLE_INPUT.glob('*') if p.is_dir()]
    for candidate in candidates:
        if (candidate / 'src' / 'eaglevision').exists() and (candidate / 'pyproject.toml').exists():
            if str(candidate).startswith('/kaggle/input'):
                dst = WORK_DIR / 'EagleVision'
                if not dst.exists():
                    print(f'Copying repo from read-only input to {dst}')
                    shutil.copytree(candidate, dst)
                return dst
            return candidate
    raise FileNotFoundError('Could not find EagleVision repo. Attach it as a Kaggle input or set REPO_DIR_OVERRIDE.')

def find_checkpoint(kind: str) -> Path | None:
    if kind == 'adapter' and ADAPTER_CHECKPOINT_OVERRIDE is not None:
        return Path(ADAPTER_CHECKPOINT_OVERRIDE)
    if kind == 'base' and BASE_DA2_CHECKPOINT_OVERRIDE is not None:
        return Path(BASE_DA2_CHECKPOINT_OVERRIDE)
    files = list(KAGGLE_INPUT.rglob('*.pt')) + list(KAGGLE_INPUT.rglob('*.pth'))
    if kind == 'adapter':
        preferred = [p for p in files if any(s in p.name.lower() for s in ['best', 'adapter', 'eaglevision'])]
        return preferred[0] if preferred else (files[0] if files else None)
    preferred = [p for p in files if any(s in p.name.lower() for s in ['depth_anything', 'da2', 'vits']) and not any(s in p.name.lower() for s in ['adapter', 'eaglevision'])]
    return preferred[0] if preferred else None

SCANNET_ROOT = find_scannet_root()
REPO_DIR = find_repo_dir()
ADAPTER_CHECKPOINT = find_checkpoint('adapter')
BASE_DA2_CHECKPOINT = find_checkpoint('base')

print('ScanNet root:', SCANNET_ROOT)
print('Repo dir:', REPO_DIR)
print('Adapter checkpoint:', ADAPTER_CHECKPOINT)
print('Base DA2 checkpoint:', BASE_DA2_CHECKPOINT)
assert ADAPTER_CHECKPOINT is not None and ADAPTER_CHECKPOINT.exists(), 'Adapter checkpoint not found. Set ADAPTER_CHECKPOINT_OVERRIDE.'


In [ ]:
os.chdir(REPO_DIR)
print('cwd:', Path.cwd())

# Kaggle usually already has torch. Editable install wires up the local CLI modules.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## Build Runtime Configs

These configs point EagleVision at the Kaggle input paths discovered above. If `BASE_DA2_CHECKPOINT` is `None`, the base model config is left with `checkpoint_path: null`; for real numbers, attach the correct DA2-Small weights or use a repo setup that already supplies them.


In [ ]:
import yaml
from copy import deepcopy

base_frozen = yaml.safe_load(Path('configs/3d_eval/da2s_frozen.yaml').read_text())
base_adapted = yaml.safe_load(Path('configs/3d_eval/da2s_eaglevision.yaml').read_text())

def patch_config(cfg):
    cfg = deepcopy(cfg)
    cfg['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
    cfg.setdefault('data', {})
    cfg['data']['root'] = str(SCANNET_ROOT)
    cfg['data']['image_size'] = IMAGE_SIZE
    cfg['data']['min_depth'] = 1e-3
    cfg['data']['max_depth'] = 10.0
    cfg['data']['depth_scale'] = 1000.0
    cfg.setdefault('base_model', {})
    cfg['base_model']['encoder'] = 'small'
    cfg['base_model']['checkpoint_path'] = str(BASE_DA2_CHECKPOINT) if BASE_DA2_CHECKPOINT is not None else None
    cfg.setdefault('eval', {})
    cfg['eval']['batch_size'] = BATCH_SIZE
    cfg['eval']['num_workers'] = NUM_WORKERS
    cfg['eval']['num_previews'] = NUM_PREVIEWS
    return cfg

frozen_cfg = patch_config(base_frozen)
adapted_cfg = patch_config(base_adapted)
FROZEN_CONFIG = CONFIG_DIR / 'da2s_frozen_kaggle.yaml'
ADAPTED_CONFIG = CONFIG_DIR / 'da2s_eaglevision_kaggle.yaml'
FROZEN_CONFIG.write_text(yaml.safe_dump(frozen_cfg, sort_keys=False), encoding='utf-8')
ADAPTED_CONFIG.write_text(yaml.safe_dump(adapted_cfg, sort_keys=False), encoding='utf-8')
print(FROZEN_CONFIG.read_text())


## Build ScanNet Frame And Pair Manifests


In [ ]:
cmd = [
    sys.executable, '-m', 'eaglevision.cli.build_rgbd_manifests',
    '--root', str(SCANNET_ROOT),
    '--out-dir', str(MANIFEST_DIR),
    '--dataset-format', 'scannet',
    '--seed', '42',
    '--max-frames-per-scene', str(MAX_FRAMES_PER_SCENE_FOR_MANIFEST),
    '--frame-stride', str(FRAME_STRIDE),
    '--max-pairs-per-scene', str(MAX_PAIRS_PER_SCENE),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print((MANIFEST_DIR / 'summary.md').read_text()[:4000])


## Dump Frozen And Adapted Depth Predictions


In [ ]:
FROZEN_PRED = PRED_DIR / 'da2s_frozen'
ADAPTED_PRED = PRED_DIR / 'da2s_eaglevision_adapted'

commands = [
    [sys.executable, '-m', 'eaglevision.cli.dump_depth_predictions', '--config', str(FROZEN_CONFIG), '--frame-manifest', str(MANIFEST_DIR / 'frames_val.jsonl'), '--method', 'frozen', '--out-dir', str(FROZEN_PRED)],
    [sys.executable, '-m', 'eaglevision.cli.dump_depth_predictions', '--config', str(ADAPTED_CONFIG), '--frame-manifest', str(MANIFEST_DIR / 'frames_val.jsonl'), '--method', 'adapted', '--checkpoint', str(ADAPTER_CHECKPOINT), '--out-dir', str(ADAPTED_PRED)],
]
for cmd in commands:
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)


## Run Core 3D-from-X Evaluators

This runs single-frame depth, pairwise RGB-D reprojection, cycle consistency, novel-view warping diagnostics, surface normals, and point cloud fusion.


In [ ]:
def run(cmd):
    print(' '.join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], check=True)

PAIR_MANIFEST = MANIFEST_DIR / 'pairs_val.jsonl'
FRAME_MANIFEST = MANIFEST_DIR / 'frames_val.jsonl'

eval_specs = [
    ('single_frame_depth', 'eaglevision.cli.eval_single_frame_depth', ['--pred-manifest'], []),
    ('cycle', 'eaglevision.cli.eval_cycle_consistency', ['--pair-manifest', PAIR_MANIFEST, '--pred-manifest'], []),
    ('reprojection', 'eaglevision.cli.eval_rgbd_reprojection', ['--pair-manifest', PAIR_MANIFEST, '--pred-manifest'], []),
    ('view_warping', 'eaglevision.cli.eval_view_warping', ['--pair-manifest', PAIR_MANIFEST, '--pred-manifest'], []),
    ('normals', 'eaglevision.cli.eval_surface_normals', ['--pred-manifest'], []),
    ('pointcloud', 'eaglevision.cli.eval_pointcloud_fusion', ['--frame-manifest', FRAME_MANIFEST, '--pred-manifest'], ['--frames-per-scene', str(FRAMES_PER_SCENE), '--max-scenes', str(MAX_SCENES)]),
]

for task, module, prefix, suffix in eval_specs:
    for label, pred_manifest in [('frozen', FROZEN_PRED / 'manifest.jsonl'), ('adapted', ADAPTED_PRED / 'manifest.jsonl')]:
        out = RAW_DIR / f'{task}_{label}.csv'
        run([sys.executable, '-m', module, *prefix, pred_manifest, '--out', out, *suffix])


## Optional Scaffold Diagnostics

These do not fabricate metrics. TSDF, mesh, pose alignment, and stereo write status rows when dependencies/data are missing.


In [ ]:
scaffold_specs = [
    ('tsdf', 'eaglevision.cli.eval_tsdf_fusion', ['--frame-manifest', FRAME_MANIFEST, '--pred-manifest'], ['--frames-per-scene', str(FRAMES_PER_SCENE)]),
    ('mesh', 'eaglevision.cli.eval_depth_to_mesh', ['--pred-manifest'], []),
    ('pose_alignment', 'eaglevision.cli.eval_pose_alignment_diagnostic', ['--pair-manifest', PAIR_MANIFEST, '--pred-manifest'], []),
    ('stereo', 'eaglevision.cli.eval_stereo_depth', ['--pred-manifest'], []),
]

for task, module, prefix, suffix in scaffold_specs:
    for label, pred_manifest in [('frozen', FROZEN_PRED / 'manifest.jsonl'), ('adapted', ADAPTED_PRED / 'manifest.jsonl')]:
        out = RAW_DIR / f'{task}_{label}.csv'
        run([sys.executable, '-m', module, *prefix, pred_manifest, '--out', out, *suffix])


## Aggregate Frozen vs Adapted Results


In [ ]:
tasks = ['single_frame_depth', 'cycle', 'reprojection', 'view_warping', 'normals', 'pointcloud']
for task in tasks:
    run([
        sys.executable, 'scripts/aggregate_3d_eval_results.py',
        '--task', task,
        '--inputs', RAW_DIR / f'{task}_frozen.csv', RAW_DIR / f'{task}_adapted.csv',
        '--baseline', 'frozen',
        '--out-dir', AGG_DIR / task,
    ])


## Quick Result Tables


In [ ]:
import pandas as pd

for task in tasks:
    table = AGG_DIR / task / 'main_table.csv'
    if table.exists():
        print('\n===', task, '===')
        display(pd.read_csv(table))

print('Raw CSVs:', RAW_DIR)
print('Aggregated summaries:', AGG_DIR)
print('Prediction previews:', FROZEN_PRED / 'previews', ADAPTED_PRED / 'previews')


## Zip Outputs For Download


In [ ]:
zip_path = WORK_DIR / 'eaglevision_3d_from_x_scannet_results.zip'
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', OUT_ROOT)
print('Wrote:', zip_path)
